In [1]:
# this is a modified version of the following script: https://github.com/atgu/bge_analysis/blob/main/tutorials/PRS_QC.ipynb

# the goal of this script is to recreate the following files with a GP missigness filter applied:  
    #'gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/all_sites_all_phenos.bed',
    #'gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/all_sites_all_phenos.bim',
    #'gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/all_sites_all_phenos.fam'

# In actuality, I did not use these files. These files were used for analysis, you can ask Lerato for the 
# scripts used to make them:
# gs://neurogap-bge-imputed-regional/lerato/wave2/plink_files/{site}_passed_all_qc.{bed,bim,fam}

In [ ]:
import hail as hl
hl.init(
    tmp_dir="gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp",
    spark_conf={'spark.driver.memory': '128g'}
)
import pandas as pd

In [3]:
# Read in manifest file
manifest = hl.import_table("gs://fc-728aff93-d8b3-439a-b002-91dd07198e77/NeuroGAP/BGE_NeuroGAP_Manifest.tsv", impute=True)
manifest.show(5)  # Display first few rows


2025-02-24 19:54:32.552 Hail: INFO: Reading table to impute column types 1) / 1]
2025-02-24 19:54:34.815 Hail: INFO: Finished type imputation        (0 + 1) / 1]
  Loading field 'PROJECT' as type str (imputed)
  Loading field 'SUBJECT_ID' as type str (imputed)
  Loading field 'DATA_TYPE' as type str (imputed)
  Loading field 'PRIMARY_DISEASE' as type str (imputed)
  Loading field 'AGE_UNSPECIFIED' as type str (imputed)
  Loading field 'RACE_ETHNICITY' as type str (imputed)
  Loading field 'DISEASE_DESCRIPTION' as type str (imputed)
  Loading field 'VERSION' as type int32 (imputed)
  Loading field 'TERRA_WORKSPACE' as type str (imputed)
  Loading field 'INVESTIGATOR' as type str (imputed)
  Loading field 'EXTERNAL_PI' as type str (imputed)
  Loading field 'COHORT' as type str (imputed)
  Loading field 'ORSP_CG' as type str (imputed)
  Loading field 'CONSENT' as type str (imputed)
  Loading field 'SEQ_ID' as type str (imputed)
  Loading field 'COLLABORATOR_PARTICIPANT_ID' as type str (im

,,,,,,,,,,,,,,,,,,,,,,,,,
PROJECT,SUBJECT_ID,DATA_TYPE,PRIMARY_DISEASE,AGE_UNSPECIFIED,RACE_ETHNICITY,DISEASE_DESCRIPTION,VERSION,TERRA_WORKSPACE,INVESTIGATOR,EXTERNAL_PI,COHORT,ORSP_CG,CONSENT,SEQ_ID,COLLABORATOR_PARTICIPANT_ID,SM_ID,PDO,CAPTURE,SEX,CRAM,GVCF,ROLE,COVERAGE_MET,CHIMERA,CONTAMINATION
str,str,str,str,str,str,str,int32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,float64,float64
"""RP-1837.PDO-29431""","""NGE0125""","""BGE""","""Bipolar Disorder""","""50""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0125""","""AAP15240340""","""SM-H5MD2""","""PDO-29431""","""broad_custom_exome_v1""","""Female""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0125/v1/NGE0125.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0125.v1.de3dd8c4-ec47-4244-ad58-567b7118f013.rb.g.vcf.gz""","""Case""","""1/31/23""",1.21e+00,1.40e-01
"""RP-1837.PDO-29431""","""NGE0134""","""BGE""","""Bipolar Disorder""","""60""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0134""","""AAP70688767""","""SM-H5MDB""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0134/v1/NGE0134.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0134.v1.4edebd6c-919e-43f5-af39-8e4f576ad580.rb.g.vcf.gz""","""Case""","""1/31/23""",1.51e+00,9.00e-02
"""RP-1837.PDO-29431""","""NGE0311""","""BGE""","""Control""","""45""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0311""","""AAP90264798""","""SM-HKAHV""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0311/v1/NGE0311.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0311.v1.54ac9b9b-02bd-47a0-9659-55280566297d.rb.g.vcf.gz""","""Control""","""4/24/23""",1.45e+00,5.00e-02
"""RP-1837.PDO-29431""","""NGE0353""","""BGE""","""Control""","""21""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0353""","""AAP19253657""","""SM-HKAJ3""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0353/v1/NGE0353.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0353.v1.fbaa384d-e980-46f4-839a-3cfc38810967.rb.g.vcf.gz""","""Control""","""1/31/23""",1.32e+00,3.00e-02
"""RP-1837.PDO-29431""","""NGE0356""","""BGE""","""Control""","""30""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0356""","""AAP20466375""","""SM-HKAJ6""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0356/v1/NGE0356.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0356.v1.3fce2de0-b9a7-4040-a09e-90adc771c569.rb.g.vcf.gz""","""Control""","""1/31/23""",1.12e+00,6.00e-02


In [4]:
# Read in duplicate IDs that were identified by `plink2 --king-cutoff 0.355`
# which was run to find the maximal independent set after removing duplicates/MZ twins
# these are the IDs we will remove first, as well as IDs identified to be "mismatches" between case and control status
# Note that this is in plink format, with FID and IID as the two columns, but these are the same ID in both columns

duplicates_to_remove_ht = hl.import_table("gs://fc-728aff93-d8b3-439a-b002-91dd07198e77/NeuroGAP/dups_to_remove.txt", impute=True, delimiter='\t')
duplicates_remove_ids = duplicates_to_remove_ht.IID.collect() #table.column_name.bring_to_local_memory
duplicates_remove_ids = hl.set(duplicates_remove_ids) # creates an array out of duplicates_remove_ids
duplicates_remove_ids.show() 

2025-02-24 19:54:37.486 Hail: INFO: Reading table to impute column types
2025-02-24 19:54:38.267 Hail: INFO: Finished type imputation
  Loading field '#FID' as type str (imputed)
  Loading field 'IID' as type str (imputed)


""
<expr>
set<str>


In [5]:
# Read in duplicates with case/control status file
duplicate_info_df = hl.import_table("gs://fc-728aff93-d8b3-439a-b002-91dd07198e77/NeuroGAP/neurogap_fullset.kin0_with_cohorts_roles_kinmin_354.csv", impute=True, delimiter=',')
duplicate_info_df.show(5)  # Display first few rows


2025-02-24 19:54:40.570 Hail: INFO: Reading table to impute column types
2025-02-24 19:54:41.303 Hail: INFO: Finished type imputation
  Loading field 'ID1' as type str (imputed)
  Loading field 'ID1_SUBJECT_ID' as type str (imputed)
  Loading field 'ID1_COLLABORATOR_PARTICIPANT_ID' as type str (imputed)
  Loading field 'ID2' as type str (imputed)
  Loading field 'ID2_SUBJECT_ID' as type str (imputed)
  Loading field 'ID2_COLLABORATOR_PARTICIPANT_ID' as type str (imputed)
  Loading field 'N_SNP' as type int32 (imputed)
  Loading field 'HetHet' as type float64 (imputed)
  Loading field 'IBS0' as type int32 (imputed)
  Loading field 'HetConc' as type float64 (imputed)
  Loading field 'HomIBS0' as type float64 (imputed)
  Loading field 'Kinship' as type float64 (imputed)
  Loading field 'IBD1Seg' as type float64 (imputed)
  Loading field 'IBD2Seg' as type float64 (imputed)
  Loading field 'PropIBD' as type float64 (imputed)
  Loading field 'InfType' as type str (imputed)
  Loading field 'I

,,,,,,,,,,,,,,,,,,,,,,,,,,
ID1,ID1_SUBJECT_ID,ID1_COLLABORATOR_PARTICIPANT_ID,ID2,ID2_SUBJECT_ID,ID2_COLLABORATOR_PARTICIPANT_ID,N_SNP,HetHet,IBS0,HetConc,HomIBS0,Kinship,IBD1Seg,IBD2Seg,PropIBD,InfType,ID1_cohort,ID2_cohort,ID1_role,ID2_role,ID1_n_times_appear_in_dup_pair,ID2_n_times_appear_in_dup_pair,ID1_primary_disease,ID2_primary_disease,same_site,same_role,same_primary_disease
str,str,str,str,str,str,int32,float64,int32,float64,float64,float64,float64,float64,float64,str,str,str,str,str,int32,int32,str,str,bool,bool,bool
"""RP-1691.PDO-29438_MAP2352""","""MAP2352""","""MAP42090985""","""RP-1691.PDO-29438_MAP1081""","""MAP1081""","""MAP80068220""",267632,6.43e-02,0,9.96e-01,1.00e-04,4.99e-01,0.00e+00,9.99e-01,9.99e-01,"""Dup/MZ""","""NeuroGAP-Psychosis_Uganda""","""NeuroGAP-Psychosis_Uganda""","""Control""","""Control""",1,1,"""Control""","""Control""",True,True,True
"""RP-1691.PDO-29469_MAP8791""","""MAP8791""","""MAP80964550""","""RP-1691.PDO-29438_MAP1092""","""MAP1092""","""MAP93579870""",272660,6.40e-02,0,9.97e-01,1.00e-04,4.99e-01,0.00e+00,9.99e-01,9.99e-01,"""Dup/MZ""","""NeuroGAP-Psychosis_Uganda""","""NeuroGAP-Psychosis_Uganda""","""Case""","""Case""",1,1,"""Bipolar Disorder""","""Schizophrenia""",True,True,False
"""RP-1691.PDO-29438_MAP1502""","""MAP1502""","""MAP94000517""","""RP-1691.PDO-29438_MAP1093""","""MAP1093""","""MAP84869921""",270414,6.43e-02,0,9.97e-01,2.00e-04,4.99e-01,0.00e+00,1.00e+00,1.00e+00,"""Dup/MZ""","""NeuroGAP-Psychosis_Uganda""","""NeuroGAP-Psychosis_Uganda""","""Case""","""Case""",1,1,"""Schizophrenia""","""Schizophrenia""",True,True,True
"""RP-1691.PDO-29469_MAP9231""","""MAP9231""","""MAP86100016""","""RP-1691.PDO-29438_MAP120""","""MAP120""","""MAP47297337""",271972,6.47e-02,0,9.97e-01,0.00e+00,4.99e-01,0.00e+00,1.00e+00,1.00e+00,"""Dup/MZ""","""NeuroGAP-Psychosis_Uganda""","""NeuroGAP-Psychosis_Uganda""","""Control""","""Control""",1,1,"""Control""","""Control""",True,True,True
"""RP-1691.PDO-29469_MAP3677""","""MAP3677""","""MAP22416927""","""RP-1691.PDO-29438_MAP1370""","""MAP1370""","""MAP61833432""",270959,6.44e-02,0,9.97e-01,0.00e+00,4.99e-01,0.00e+00,1.00e+00,1.00e+00,"""Dup/MZ""","""NeuroGAP-Psychosis_Uganda""","""NeuroGAP-Psychosis_Uganda""","""Case""","""Case""",2,2,"""Schizophrenia""","""Schizophrenia""",True,True,True


In [6]:
# Subset the duplicates table to only include instances where diagnosis does not match (same_primary_disease == False)
mismatches = duplicate_info_df.filter(duplicate_info_df.same_primary_disease == False) # .filter filters rows of duplicate_info_df based on the condition that same_primary_disease == False to retain these rows
#note: could also subset by same_role==False if bipolar/SCZ status are grouped together

In [7]:
# Filter duplicates table to exclude mismatches (for both ID1 and ID2)

# Collect the mismatching IDs into Python lists
mismatch_id1_list = mismatches.ID1_SUBJECT_ID.collect()
mismatch_id2_list = mismatches.ID2_SUBJECT_ID.collect()

# Convert them to Hail sets
mismatch_id1_set = hl.set(mismatch_id1_list)
mismatch_id2_set = hl.set(mismatch_id2_list)

# combine with set of IDs to remove from plink2 command
combined_set = mismatch_id1_set.union(mismatch_id2_set).union(duplicates_remove_ids)
# combines mismatch_id1_set, mismatch_id2_set, and duplicates_remove_ids into one dataset


In [8]:
hl.eval(hl.len(combined_set)) #to check how many duplicates / mismatched duplicates are in the set

943

In [9]:
# Get a list of all included projects in the manifest
projects = manifest.TERRA_WORKSPACE.collect()
print("Projects in the manifest:", set(projects))

Projects in the manifest: {'NeuroGAP-Psychosis_Uganda_Psychosis_BGE', 'NeuroGAP-Psychosis_AAU_Psychosis_BGE', 'NeuroGAP-Psychosis_KEMRI_Psychosis_BGE', 'NeuroGAP-Psychosis_Moi_Psychosis_BGE', 'NeuroGAP-Psychosis_UCT_Psychosis_BGE'}


In [10]:
# Subset the individual manifest files to include all sites

# Uganda
manifest_sub_uganda = manifest.filter(manifest.TERRA_WORKSPACE == "NeuroGAP-Psychosis_Uganda_Psychosis_BGE")
manifest_sub_uganda = manifest_sub_uganda.annotate(SITE="Uganda")

# KEMRI
manifest_sub_kemri = manifest.filter(manifest.TERRA_WORKSPACE == "NeuroGAP-Psychosis_KEMRI_Psychosis_BGE")
manifest_sub_kemri = manifest_sub_kemri.annotate(SITE="KEMRI")

# Moi
manifest_sub_moi = manifest.filter(manifest.TERRA_WORKSPACE == "NeuroGAP-Psychosis_Moi_Psychosis_BGE")
manifest_sub_moi = manifest_sub_moi.annotate(SITE="Moi")

# UCT South Africa
manifest_sub_uct = manifest.filter(manifest.TERRA_WORKSPACE == "NeuroGAP-Psychosis_UCT_Psychosis_BGE")
manifest_sub_uct = manifest_sub_uct.annotate(SITE="UCT")

# Ethiopia - AAU
manifest_sub_aau = manifest.filter(manifest.TERRA_WORKSPACE == "NeuroGAP-Psychosis_AAU_Psychosis_BGE")
manifest_sub_aau = manifest_sub_aau.annotate(SITE="AAU")

In [11]:
# Combine all tables into one
manifest_sub_all = (
    manifest_sub_aau.union(manifest_sub_uganda)
    .union(manifest_sub_kemri)
    .union(manifest_sub_moi)
    .union(manifest_sub_uct)
)

# Show the result to verify
manifest_sub_all.show()

,,,,,,,,,,,,,,,,,,,,,,,,,,
PROJECT,SUBJECT_ID,DATA_TYPE,PRIMARY_DISEASE,AGE_UNSPECIFIED,RACE_ETHNICITY,DISEASE_DESCRIPTION,VERSION,TERRA_WORKSPACE,INVESTIGATOR,EXTERNAL_PI,COHORT,ORSP_CG,CONSENT,SEQ_ID,COLLABORATOR_PARTICIPANT_ID,SM_ID,PDO,CAPTURE,SEX,CRAM,GVCF,ROLE,COVERAGE_MET,CHIMERA,CONTAMINATION,SITE
str,str,str,str,str,str,str,int32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,float64,float64,str
"""RP-1837.PDO-29431""","""NGE0125""","""BGE""","""Bipolar Disorder""","""50""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0125""","""AAP15240340""","""SM-H5MD2""","""PDO-29431""","""broad_custom_exome_v1""","""Female""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0125/v1/NGE0125.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0125.v1.de3dd8c4-ec47-4244-ad58-567b7118f013.rb.g.vcf.gz""","""Case""","""1/31/23""",1.21e+00,1.40e-01,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0134""","""BGE""","""Bipolar Disorder""","""60""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0134""","""AAP70688767""","""SM-H5MDB""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0134/v1/NGE0134.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0134.v1.4edebd6c-919e-43f5-af39-8e4f576ad580.rb.g.vcf.gz""","""Case""","""1/31/23""",1.51e+00,9.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0311""","""BGE""","""Control""","""45""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0311""","""AAP90264798""","""SM-HKAHV""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0311/v1/NGE0311.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0311.v1.54ac9b9b-02bd-47a0-9659-55280566297d.rb.g.vcf.gz""","""Control""","""4/24/23""",1.45e+00,5.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0353""","""BGE""","""Control""","""21""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0353""","""AAP19253657""","""SM-HKAJ3""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0353/v1/NGE0353.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0353.v1.fbaa384d-e980-46f4-839a-3cfc38810967.rb.g.vcf.gz""","""Control""","""1/31/23""",1.32e+00,3.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0356""","""BGE""","""Control""","""30""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0356""","""AAP20466375""","""SM-HKAJ6""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0356/v1/NGE0356.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0356.v1.3fce2de0-b9a7-4040-a09e-90adc771c569.rb.g.vcf.gz""","""Control""","""1/31/23""",1.12e+00,6.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0367""","""BGE""","""Schizophrenia""","""43""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE036

In [17]:
# isolate SUBJECT_ID (aka, IID) and SITE for analyzing data later by site

# Select only the IID (SUBJECT_ID) and SITE columns
iid_site_table = manifest_sub_all.select('SUBJECT_ID', 'SITE')

# Rename SUBJECT_ID to IID 
iid_site_table = iid_site_table.rename({'SUBJECT_ID': 'IID'})

# Show the table to verify
iid_site_table.show(5)


,
IID,SITE
str,str
"""NGE0125""","""AAU"""
"""NGE0134""","""AAU"""
"""NGE0311""","""AAU"""
"""NGE0353""","""AAU"""
"""NGE0356""","""AAU"""


In [18]:
# Download the file to the indicated google cloud storage

# Define Google Cloud Storage output path
output_path = "gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/IID_SITE.tsv"

# Export the IID-SITE table
iid_site_table.export(output_path, header=True)

2025-02-24 20:12:43.081 Hail: INFO: merging 6 files totalling 629.0K...+ 5) / 5]
2025-02-24 20:12:43.797 Hail: INFO: while writing:
    gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/IID_SITE.tsv
  merge time: 715.819ms


In [12]:
# print all column names names, want to see if SITE is added

# Print the column names of the combined manifest
print(list(manifest_sub_all.row))


['PROJECT', 'SUBJECT_ID', 'DATA_TYPE', 'PRIMARY_DISEASE', 'AGE_UNSPECIFIED', 'RACE_ETHNICITY', 'DISEASE_DESCRIPTION', 'VERSION', 'TERRA_WORKSPACE', 'INVESTIGATOR', 'EXTERNAL_PI', 'COHORT', 'ORSP_CG', 'CONSENT', 'SEQ_ID', 'COLLABORATOR_PARTICIPANT_ID', 'SM_ID', 'PDO', 'CAPTURE', 'SEX', 'CRAM', 'GVCF', 'ROLE', 'COVERAGE_MET', 'CHIMERA', 'CONTAMINATION', 'SITE']


In [13]:
#read in the BGEN files (all autosomes)

input_path = f"gs://fc-728aff93-d8b3-439a-b002-91dd07198e77/NeuroGAP/bgen/neurogap_final_merged_INFO0.8_rsIDs_chr*.bgen"

bgen = hl.import_bgen(input_path,entry_fields=['GT','GP']) # imports a bgen file from the indicated input path with the entry fields GT and GP
# GT = hard-called genotype
# GP = probabilistic genotype likelihoods for different genotypes


2025-02-24 19:59:36.184 Hail: INFO: Number of BGEN files parsed: 22
2025-02-24 19:59:36.185 Hail: INFO: Number of samples in BGEN files: 38797
2025-02-24 19:59:36.185 Hail: INFO: Number of variants across all BGEN files: 32294864


In [14]:
all_sample_ids = bgen.col_key.s.collect()
# col_key represents primary identifier for columns
# .s is the sample ID
# .collect gathers all sample IDs (from .s) into a python list

2025-02-24 20:00:51.955 Hail: INFO: Number of BGEN files parsed: 22
2025-02-24 20:00:51.955 Hail: INFO: Number of samples in BGEN files: 38797
2025-02-24 20:00:51.955 Hail: INFO: Number of variants across all BGEN files: 32294864


In [15]:
# prints the ount of sample ids from the BGEN files (all autosomes)

len(all_sample_ids)

38797

In [16]:
# Subset the main sample list using the subsetted manifest IDs
# Collect the subsetted manifest IDs into a Python list
manifest_sub_ids = manifest_sub_all.SUBJECT_ID.collect()

# Filter all_sample_ids to include only those that are in the manifest_sub_ids
filtered_sample_ids = [sample for sample in all_sample_ids if sample in manifest_sub_ids]
# itereates through all_sample_ids
    # keeps only those that exist in manifest_sub_ids


In [18]:
# Evaluate the combined_set into a Python set
combined_set_py = hl.eval(combined_set)
# .eval prints the nature of a dataset --> checks datatype

# Convert the Python set to a list
combined_list = list(combined_set_py)


# Filter filtered_sample_ids to exclude any samples that are in the combined duplicate set
final_sample_ids = [sample for sample in filtered_sample_ids
                    if sample not in combined_list]


In [19]:
# Convert the final_sample_ids list into a Hail set
final_sample_set = hl.set(final_sample_ids)

# Filter the MatrixTable to keep only the columns (samples) present in final_sample_ids
bgen_subset = bgen.filter_cols(final_sample_set.contains(bgen.s))
# filter_cols(#condition) filters columns based on the indicated condition
    # based on bgen.s sample IDs
        # .contains checks if current sample ID is in final_sample_set


In [20]:
#checkpoint the subset matrixTable so it doesn't need to be re-computed
bgen_subset = bgen_subset.checkpoint('gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp/all_prs_bgen_subset1.mt', overwrite=True)

2025-02-17 20:42:06.873 Hail: INFO: Number of BGEN files parsed: 22
2025-02-17 20:42:06.874 Hail: INFO: Number of samples in BGEN files: 38797
2025-02-17 20:42:06.874 Hail: INFO: Number of variants across all BGEN files: 32294864
2025-02-17 21:42:26.115 Hail: INFO: wrote matrix table with 32294864 rows and 37846 columns in 1087 partitions to gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp/all_prs_bgen_subset1.mt


In [21]:
#get variant QC statistics
bgen_subset = hl.variant_qc(bgen_subset)

In [22]:
#load in precomputed KING file with kinship values for pairs of related samples
relatedness_ht = hl.import_table("gs://fc-728aff93-d8b3-439a-b002-91dd07198e77/NeuroGAP/neurogap_imputed_king_kinship_no_dup.txt", impute=True)
relatedness_ht.show()

2025-02-17 21:42:28.669 Hail: INFO: Reading table to impute column types
2025-02-17 21:42:29.189 Hail: INFO: Finished type imputation
  Loading field 'ID1' as type str (imputed)
  Loading field 'ID2' as type str (imputed)
  Loading field 'Kinship' as type float64 (imputed)


,,
ID1,ID2,Kinship
str,str,float64
"""NGP6484-1KTM""","""NGP_3109-1TTG""",2.50e-01
"""MAP11066""","""MAP11833""",2.47e-01
"""NGP8956-1BVQ""","""NGP4041-1VUI""",9.22e-02
"""NGE7298_2""","""NGE7683""",2.46e-01
"""NGP_1716-1TVT""","""NGP6236-1BFM""",1.23e-01
"""MAP6971""","""MAP2398""",2.46e-01
"""NGP968-1WJJ""","""NGP_1185-1HMM""",2.46e-01
"""NGP8622-1OZP""","""NGP4251-1DWJ""",2.34e-01


In [23]:
#subset KING kinship file to just the individuals in your subset
relatedness_ht_sub = relatedness_ht.filter(
    final_sample_set.contains(relatedness_ht.ID1) |
    final_sample_set.contains(relatedness_ht.ID2)
)

In [24]:
related_samples = hl.maximal_independent_set(relatedness_ht_sub.ID1, relatedness_ht_sub.ID2, False) 
# .maxmimal_independent_set is used to find subset of individuals where no two members are related
    # based on relatedness_ht_sub.ID1 & relatedness_ht_sub.ID2
        # False indicates that we want to remove related samples and keep a maximally independent subset

2025-02-17 21:43:15.628 Hail: INFO: wrote table with 1948 rows in 1 partition to gs://neurogap-bge-imputed-regional/nico/khat_gwas/tmp/eGheRj5Xwqnvs2TgGKPZZZ


In [25]:
#subset again to just the unrelated individuals
bgen_subset = bgen_subset.filter_cols(hl.is_defined(related_samples[bgen_subset.col_key]), keep=False) 
# removes related samples from bgen_subset by checking if they exist in related_samples

In [26]:
#get variant QC statistics
bgen_subset = hl.variant_qc(bgen_subset)

In [27]:
#get the MAC per variant which is just the minimum of the AC
bgen_subset = bgen_subset.annotate_rows(MAC = hl.min(bgen_subset.variant_qc.AC[1:]))
# adds a new annotation (MAC) to each variant in bgen_subset
    # computing minor allele count from AC values
# .annotate_rows(MAC = hl.min(AC[1:])) computes MAC




In [28]:
#remove alleles with MAC<20
bgen_filtered = bgen_subset.filter_rows(bgen_subset.MAC >= 20)
# with the MAC, then removes those with MAC >=20

# print("Number of SNPs flagged for MAC filter:", num_snps_bgen_filtered_MAC)


In [29]:
#set any samples with a variant that has max(GP)<1.0 to missing
bgen_filtered_GP = bgen_filtered.annotate_entries(
    GT = hl.or_missing(
        hl.max(bgen_filtered.GP) == 0.8,  # Keep GT if max(GP) == 1.0, otherwise set to missing
        bgen_filtered.GT  # If condition is met, keep original GT
    )
)

In [32]:
#checkpoint the subset matrixTable so it doesn't need to be re-computed
bgen_prs_subset = bgen_filtered_GP.checkpoint('gs://neurogap-bge-imputed-regional/temp/all_sites_prs_bgen_subset2.mt', overwrite=True)


2025-02-17 21:43:19.145 Hail: INFO: Ordering unsorted dataset with network shuffle
2025-02-17 23:26:28.609 Hail: INFO: wrote matrix table with 27585667 rows and 36299 columns in 1087 partitions to gs://neurogap-bge-imputed-regional/temp/all_sites_prs_bgen_subset2.mt


In [33]:
# converting special characters
# remove all spaces, underscores, and dots and replace them with dashes "-"

samples = bgen_prs_subset["s"].collect()                                                                                                      
samples_new = [sample.replace(" ", "-") for sample in samples]                                                                          
samples_new = [sample.replace("_", "-") for sample in samples_new]
samples_new = [sample.replace(".", "-") for sample in samples_new]
                                                                                                                            
pandas_df = pd.DataFrame({"s":samples,"s_new":samples_new})                                                                       

                                                                                                                       
ht=hl.Table.from_pandas(pandas_df)                                                                                              
ht=ht.key_by(ht.s)  

 

In [34]:
bgen_prs_subset = bgen_prs_subset.annotate_cols(s_new = ht[bgen_prs_subset.s].s_new) 

In [35]:
ids = bgen_prs_subset.col_key.s.collect()

In [36]:
manifest_prs_sub = manifest_sub_all.filter(hl.literal(ids).contains(manifest_sub_all.SUBJECT_ID))
manifest_prs_sub.show(5)

,,,,,,,,,,,,,,,,,,,,,,,,,,
PROJECT,SUBJECT_ID,DATA_TYPE,PRIMARY_DISEASE,AGE_UNSPECIFIED,RACE_ETHNICITY,DISEASE_DESCRIPTION,VERSION,TERRA_WORKSPACE,INVESTIGATOR,EXTERNAL_PI,COHORT,ORSP_CG,CONSENT,SEQ_ID,COLLABORATOR_PARTICIPANT_ID,SM_ID,PDO,CAPTURE,SEX,CRAM,GVCF,ROLE,COVERAGE_MET,CHIMERA,CONTAMINATION,SITE
str,str,str,str,str,str,str,int32,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,float64,float64,str
"""RP-1837.PDO-29431""","""NGE0125""","""BGE""","""Bipolar Disorder""","""50""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0125""","""AAP15240340""","""SM-H5MD2""","""PDO-29431""","""broad_custom_exome_v1""","""Female""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0125/v1/NGE0125.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0125.v1.de3dd8c4-ec47-4244-ad58-567b7118f013.rb.g.vcf.gz""","""Case""","""1/31/23""",1.21e+00,1.40e-01,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0134""","""BGE""","""Bipolar Disorder""","""60""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0134""","""AAP70688767""","""SM-H5MDB""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0134/v1/NGE0134.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0134.v1.4edebd6c-919e-43f5-af39-8e4f576ad580.rb.g.vcf.gz""","""Case""","""1/31/23""",1.51e+00,9.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0353""","""BGE""","""Control""","""21""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0353""","""AAP19253657""","""SM-HKAJ3""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0353/v1/NGE0353.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0353.v1.fbaa384d-e980-46f4-839a-3cfc38810967.rb.g.vcf.gz""","""Control""","""1/31/23""",1.32e+00,3.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0356""","""BGE""","""Control""","""30""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0356""","""AAP20466375""","""SM-HKAJ6""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0356/v1/NGE0356.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0356.v1.3fce2de0-b9a7-4040-a09e-90adc771c569.rb.g.vcf.gz""","""Control""","""1/31/23""",1.12e+00,6.00e-02,"""AAU"""
"""RP-1837.PDO-29431""","""NGE0367""","""BGE""","""Schizophrenia""","""43""","""African""","""SCZ_BP""",1,"""NeuroGAP-Psychosis_AAU_Psychosis_BGE""","""Koenen""","""Teferra""","""NeuroGAP-Psychosis_AAU""","""CG-5600""","""MDS""","""RP-1837.PDO-29431_NGE0367""","""AAP33014231""","""SM-HKAJH""","""PDO-29431""","""broad_custom_exome_v1""","""Male""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/NeuroGAP-Psychosis_AAU_Psychosis_BGE/RP-1837.PDO-29431/Exome/NGE0367/v1/NGE0367.cram""","""gs://fc-8bcb1af7-a219-4671-90f7-e5653e87c3ad/AAU_gVCF/gvcfs/NGE0367.v1.f66ffe63-539f-4483-af97-fb9c603e6c53.rb.g.vcf.gz""","""Case""","""1/31/23""",1.28e+00,4.00e-02,"""AAU"""


Everyhing marked # text export: ... is subsetting a random 1k variants from bgen_prs_subset for all 5 sites (same varsity by site)

The goal is to just test on a small subset before we export all sites

In [ ]:
# TEST EXPORT: subsetting 1k random variants for each site
mt_with_rand = bgen_prs_subset.annotate_rows(rand = hl.rand_unif(0, 1))

# Sort rows by random value and take first 1000
mt_1k_all = mt_with_rand._select_rows(
    row_exprs=mt_with_rand.row_value,  # preserve row fields
    entries=mt_with_rand.entry,
).key_rows_by("rand").order_rows_by("rand").head(1000, bgen_prs_subset.count_cols())


In [ ]:
# TEST EXPORT: subsetting 1k random variants for each site
def varid_expr(mt):
    return hl.or_else(
        mt.rsid,
        hl.format("%s:%s:%s", mt.locus, mt.alleles[0], mt.alleles[1])
    )

for site in sites:
    # --- Gather SUBJECT_IDs for this site from your manifest ---
    site_ids = manifest_prs_sub.filter(manifest_prs_sub.SITE == site) \
                               .aggregate(hl.agg.collect(manifest_prs_sub.SUBJECT_ID))
    if not site_ids:
        print(f"[{site}] No SUBJECT_IDs found in manifest, skipping TEST_1k.")
        continue

    id_set = hl.literal(set(site_ids))

    mt_1k_site = mt_1k_all.filter_cols(id_set.contains(mt_1k_all.s_new))
    n_cols = mt_1k_site.count_cols()
    if n_cols == 0:
        print(f"[{site}] 0 samples in TEST_1k after filtering, skipping.")
        continue


In [ ]:
# TEST EXPORT: subsetting 1k random variants for each site
plink_path_test = (
        f"gs://neurogap-bge-imputed-regional/nico/khat_gwas/"
        f"gp_missingness_dataset/gp08/adjusted_plink_filters/by_site/"
        f"NeuroGAP_{site}_TEST_1k"
    )

hl.export_plink(
        mt_1k_site,
        output=plink_path_test,
        fam_id=mt_1k_site.s_new,
        ind_id=mt_1k_site.s_new,
        varid=varid_expr(mt_1k_site)
    )

The code below marked "# EXPORT BY SITE: ..." is labeled to export the NeuroGAP_gp08 data by site
Will be using this as a test code for POLMM

In [ ]:
# EXPORT BY SITE: Process & subset by site and get counts

for site in sites:
    # subset manifest to this site
    site_ids = manifest_prs_sub.filter(manifest_prs_sub.SITE == site) \
                               .aggregate(hl.agg.collect(manifest_prs_sub.SUBJECT_ID))

    if not site_ids:
        print(f"[{site}] No SUBJECT_IDs found, skipping.")
        continue

    # filter matrix table columns by those IDs
    mt_site = bgen_prs_subset.filter_cols(
        hl.literal(set(site_ids)).contains(bgen_prs_subset.s_new)
    )

    n_cols = mt_site.count_cols()
    if n_cols == 0:
        print(f"[{site}] 0 samples found in MT for {site}, skipping.")
        continue

In [39]:
# EXPORT BY SITE: begin sending to plink path

plink_path="gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/gp08/adjusted_plink_filters/by_site/NeuroGAP_{$SITE}"                         

hl.export_plink(bgen_prs_subset,output=plink_path,fam_id=bgen_prs_subset.s_new, ind_id=bgen_prs_subset.s_new,varid=bgen_prs_subset["rsid"]) 

# check if bgen_prs_subset.s_new and rsid exist

2025-02-17 23:56:56.221 Hail: INFO: merging 1088 files totalling 233.1G...) / 4]
2025-02-18 01:00:52.430 Hail: INFO: while writing:==================(4 + 1) / 4]
    gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/neurogap_all_sites_v2.bed
  merge time: 1h3m56.2s
2025-02-18 01:01:15.542 Hail: INFO: merging 1087 files totalling 897.1M...
2025-02-18 01:03:08.806 Hail: INFO: while writing:
    gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/neurogap_all_sites_v2.bim
  merge time: 1m53.3s
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/miniconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/conda/miniconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/conda/miniconda3/lib/python3.11/si

In [ ]:
# ORIGINAL EXPORT: Just aggregates everything into one big file. 
plink_path="gs://neurogap-bge-imputed-regional/nico/khat_gwas/gp_missingness_dataset/neurogap_all_sites_v2\ 

hl.export_plink(bgen_prs_subset,output=plink_path,fam_id=bgen_prs_subset.s_new, ind_id=bgen_prs_subset.s_new,varid=bgen_prs_subset["rsid"])